# Phase 2: Credit Feature Engineering — Weight of Evidence (WoE) & Information Value (IV)
## Section: Traditional Credit Risk Binning Architecture

### 1. Objective
This notebook handles the transformation of raw continuous and categorical features into credit-standard inputs. Traditional scorecards do not ingest raw numbers; they rely on **Weight of Evidence (WoE)** transformations to linearize relationships and isolate risk.

### 2. The Mathematics of Credit Scoring
To ensure model stability, handle outliers natively, and maximize regulatory transparency, features are binned, and each bin is transformed using WoE:

$$WoE = \ln\left(\frac{\% \text{ Distribution of Good Borrowers}}{\% \text{ Distribution of Bad Borrowers}}\right)$$

* A **positive WoE** indicates a category has a higher concentration of safe borrowers (reduces probability of default).
* A **negative WoE** indicates a high-concentration risk pocket (increases probability of default).

### 3. Feature Selection via Information Value (IV)
To determine if a variable provides enough predictive power to enter the model, we calculate its **Information Value (IV)**:

$$IV = \sum \left( \% \text{ Distribution of Goods} - \% \text{ Distribution of Bads} \right) \times WoE$$

#### Industry Selection Benchmarks:
* $< 0.02$: Unpredictive (Drop)
* $0.02 \text{ to } 0.10$: Weak Predictor (Keep if stable, e.g., Annual Income)
* $0.10 \text{ to } 0.30$: Medium Predictor
* $0.30 \text{ to } 0.50$: Strong Predictor
* $> 0.50$: Suspiciously High / Potential Data Leak (Requires deep audit)

### 4. Monotonicity Constraints
Continuous variables (`annual_inc`, `int_rate`, `loan_amnt`) are monitored to ensure their WoE values follow a logical, monotonic trend. This guarantees the model complies with internal risk policies and external banking audits. Engineered data frames are saved as highly optimized Parquet files for model training.

In [1]:
import os
import pandas as pd
import numpy as np

PROCESSED_DATA_DIR = "data/processed"
input_path = os.path.join(PROCESSED_DATA_DIR, "01_cleaned_cohort.csv.gz")
df_master = pd.read_csv(input_path, compression='gzip')

print(f"Master performance cohort reloaded. Shape: {df_master.shape}")

Master performance cohort reloaded. Shape: (1119711, 51)


In [2]:
# ── Leakage & redundancy controls ─────────────────────────────────────────────
# LEAKAGE: these fields are realised only AFTER a loan defaults / runs to term.
# They are not known at the application decision point, so they must NEVER enter a
# PD model. (They are legitimately used to BUILD the LGD/EAD targets in notebook 05,
# which reads the raw cohort directly — not these WoE features.)
#   total_rec_prncp — cumulative principal repaid to date (IV>2 ⇒ textbook leakage)
#   recoveries      — post charge-off recovery cash flow
leakage_cols = ['total_rec_prncp', 'recoveries']

# REDUNDANCY: drop near-perfect duplicates / deterministic derivatives to avoid the
# unstable, exploding coefficients that multicollinearity produces in a WoE logit.
#   funded_amnt — equals loan_amnt on ~all records
#   installment — deterministic function of loan_amnt, int_rate, term (weak IV anyway)
redundant_cols = ['funded_amnt', 'installment']

exclude_cols = ['target', 'loan_status'] + leakage_cols + redundant_cols
features_to_bin = [col for col in df_master.columns if col not in exclude_cols]

# CRITICAL — label-consistency contract:
# We derive quantile cut-points with qcut(retbins=True) but materialise the bin column
# with pd.cut(bins=edges, include_lowest=True). Every downstream consumer (notebook 06
# OOT scoring/PSI and app.py inference) re-bins new data with the *same* pd.cut call, so
# the interval-label strings are byte-identical and WoE look-ups never silently miss.
# (Using qcut's own precision-rounded labels here would not match pd.cut on the raw
# edges, causing non-round features like int_rate/dti to default to WoE 0.0 at inference.)
bin_edges = {}
for col in features_to_bin:
    if df_master[col].dtype in [np.float64, np.int64]:
        try:
            _, edges = pd.qcut(df_master[col], q=5, duplicates='drop', retbins=True)
        except ValueError:
            _, edges = pd.cut(df_master[col], bins=5, retbins=True)
        bin_edges[col] = edges
        df_master[f'bin_{col}'] = pd.cut(df_master[col], bins=edges, include_lowest=True).astype(str)
    else:
        df_master[f'bin_{col}'] = df_master[col].astype(str)

print(f"Excluded from PD features  : {leakage_cols} (leakage) + {redundant_cols} (redundant)")
print(f"Coarse bin partitions established across {len(features_to_bin)} features.")

Excluded from PD features  : ['total_rec_prncp', 'recoveries'] (leakage) + ['funded_amnt', 'installment'] (redundant)
Coarse bin partitions established across 45 features.


In [3]:
def calculate_woe_iv(df, feature_col, target_col):
    total_bads = df[target_col].sum()
    total_goods = len(df) - total_bads

    grouped = df.groupby(feature_col)[target_col].agg(['count', 'sum']).reset_index()
    grouped.columns = [feature_col, 'total', 'bads']
    grouped['goods'] = grouped['total'] - grouped['bads']

    grouped['dist_goods'] = grouped['goods'] / total_goods
    grouped['dist_bads'] = grouped['bads'] / total_bads
    grouped['dist_goods'] = grouped['dist_goods'].replace(0, 0.0001)
    grouped['dist_bads'] = grouped['dist_bads'].replace(0, 0.0001)

    grouped['WoE'] = np.log(grouped['dist_goods'] / grouped['dist_bads'])
    grouped['IV_contribution'] = (grouped['dist_goods'] - grouped['dist_bads']) * grouped['WoE']

    total_iv = grouped['IV_contribution'].sum()
    mapping_dict = dict(zip(grouped[feature_col], grouped['WoE']))
    return mapping_dict, total_iv, grouped


# NEW: parse left bound of interval string "(10.0, 15.0]" → 10.0, handles "-inf"
def bin_sort_key(label):
    left = label.strip().lstrip('([').split(',')[0].strip()
    return float('-inf') if left == '-inf' else float(left)


columns_to_process = [col for col in df_master.columns if col.startswith('bin_') and not col.endswith('_WoE')]
pipeline_mappings = {}

print("=== CALCULATING PORTFOLIO INFORMATION VALUE (IV) & PROGRAMMATIC SCREENING ===")
for col in columns_to_process:
    mapping_dict, total_iv, summary_table = calculate_woe_iv(df_master, col, 'target')

    df_master[f"{col}_WoE"] = df_master[col].map(mapping_dict).fillna(0.0)

    if total_iv >= 0.02:
        print(f"Feature: {col:<35} | IV: {total_iv:.4f} | Status: RETAINED")
        pipeline_mappings[col] = mapping_dict

        orig_name = col.replace('bin_', '')
        if orig_name in ['annual_inc', 'int_rate', 'loan_amnt', 'dti', 'fico_range_low']:
            # FIXED: sort by numeric left bound, not alphabetically
            woe_values = [mapping_dict[b] for b in sorted(mapping_dict.keys(), key=bin_sort_key)]
            differences = np.diff(woe_values)
            is_monotonic = np.all(differences >= 0) or np.all(differences <= 0)
            if not is_monotonic:
                print(f"   ⚠️ AUDIT WARNING: '{orig_name}' violates strict monotonicity constraints!")
    else:
        print(f"Feature: {col:<35} | IV: {total_iv:.4f} | Status: DROPPED (Low Signal)")
        df_master.drop(columns=[f"{col}_WoE"], inplace=True)

# NEW: fico_range_high differs from fico_range_low by exactly 4 points on every Lending Club record.
# Keeping both causes near-perfect multicollinearity in the logistic regression.
df_master.drop(columns=['bin_fico_range_high_WoE'], errors='ignore', inplace=True)
pipeline_mappings.pop('bin_fico_range_high', None)
print("Dropped bin_fico_range_high_WoE: near-duplicate of fico_range_low (collinearity)")

=== CALCULATING PORTFOLIO INFORMATION VALUE (IV) & PROGRAMMATIC SCREENING ===
Feature: bin_loan_amnt                       | IV: 0.0314 | Status: RETAINED


Feature: bin_term                            | IV: 0.1972 | Status: RETAINED
Feature: bin_int_rate                        | IV: 0.4231 | Status: RETAINED


Feature: bin_grade                           | IV: 0.4677 | Status: RETAINED
Feature: bin_sub_grade                       | IV: 0.5019 | Status: RETAINED


Feature: bin_emp_length                      | IV: 0.0014 | Status: DROPPED (Low Signal)
Feature: bin_home_ownership                  | IV: 0.0265 | Status: RETAINED


Feature: bin_annual_inc                      | IV: 0.0290 | Status: RETAINED
Feature: bin_verification_status             | IV: 0.0532 | Status: RETAINED


...

Feature: bin_bc_util                         | IV: 0.0224 | Status: RETAINED
Feature: bin_chargeoff_within_12_mths        | IV: 0.0000 | Status: DROPPED (Low Signal)


Feature: bin_delinq_amnt                     | IV: 0.0000 | Status: DROPPED (Low Signal)
Feature: bin_mo_sin_old_il_acct              | IV: 0.0063 | Status: DROPPED (Low Signal)


Feature: bin_mo_sin_old_rev_tl_op            | IV: 0.0214 | Status: RETAINED
Feature: bin_mo_sin_rcnt_rev_tl_op           | IV: 0.0262 | Status: RETAINED


Feature: bin_mo_sin_rcnt_tl                  | IV: 0.0312 | Status: RETAINED
Feature: bin_mort_acc                        | IV: 0.0261 | Status: RETAINED


Feature: bin_num_accts_ever_120_pd           | IV: 0.0009 | Status: DROPPED (Low Signal)
Feature: bin_num_actv_rev_tl                 | IV: 0.0289 | Status: RETAINED
Dropped bin_fico_range_high_WoE: near-duplicate of fico_range_low (collinearity)


In [4]:
output_path = os.path.join(PROCESSED_DATA_DIR, "02_engineered_features.csv.gz")
df_master.to_csv(output_path, index=False, compression='gzip')

import pickle
with open(os.path.join(PROCESSED_DATA_DIR, "woe_mappings.pkl"), "wb") as f:
    pickle.dump(pipeline_mappings, f)

# NEW: bin_edges must be saved so app.py and any batch scoring job can replicate
# the exact same binning on new applicant data before applying WoE lookup
with open(os.path.join(PROCESSED_DATA_DIR, "bin_edges.pkl"), "wb") as f:
    pickle.dump(bin_edges, f)

print("Feature engineering stage finalized. Target csv references locked.")

Feature engineering stage finalized. Target csv references locked.
